# Salary Prediction with Linear Regression
## Jupyter Notebook + Gradio UI + Flask UI

This project uses `Salary_dataset.csv` to predict **Salary** from **YearsExperience** with Linear Regression. It includes model training, evaluation, a plot, and two user interfaces (Gradio and Flask).

**Dataset columns:** `YearsExperience` is the input feature; `Salary` is the target. The `Unnamed: 0` column is an index and is not used for training.

## 1. Install libraries

In [ ]:
%pip install -q pandas numpy matplotlib scikit-learn gradio flask

## 2. Imports and load dataset

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_PATH = "Salary_dataset.csv"

# If the CSV is not beside this notebook, change DATA_PATH to its full path.
if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        uploaded = files.upload()
        DATA_PATH = next((name for name in uploaded if name.lower().endswith(".csv")), DATA_PATH)
    except Exception:
        pass

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError("Salary_dataset.csv not found. Place it in the notebook folder or upload it.")

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()
display(df.head())
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())


## 3. Prepare data and train Linear Regression

In [ ]:
# Use only the meaningful input feature and target; ignore the index column.
X = df[["YearsExperience"]].apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(df["Salary"], errors="coerce")

clean = pd.concat([X, y.rename("Salary")], axis=1).dropna()
X = clean[["YearsExperience"]]
y = clean["Salary"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Intercept: {model.intercept_:,.2f}")
print(f"Coefficient (slope): {model.coef_[0]:,.2f}")
print(f"MAE:  {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R²:   {r2:.4f}")
print(f"Equation: Salary = {model.intercept_:,.2f} + ({model.coef_[0]:,.2f} × YearsExperience)")


## 4. Visualize the regression line

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(X["YearsExperience"], y, label="Actual data")
line_x = np.linspace(X["YearsExperience"].min(), X["YearsExperience"].max(), 100)
line_y = model.predict(pd.DataFrame({"YearsExperience": line_x}))
plt.plot(line_x, line_y, label="Linear regression line")
plt.xlabel("Years of Experience")
plt.ylabel("Salary")
plt.title("Salary Prediction using Linear Regression")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 5. Prediction helper

In [ ]:
def predict_salary(years_experience):
    years = float(years_experience)
    if not np.isfinite(years) or years < 0:
        raise ValueError("Years of experience must be a non-negative number.")
    prediction = float(model.predict(pd.DataFrame({"YearsExperience": [years]}))[0])
    return round(prediction, 2)

print("Example prediction for 5 years:", predict_salary(5))


## 6. Gradio user interface

In [ ]:
import gradio as gr

def gradio_predict(years_experience):
    if years_experience is None:
        raise gr.Error("Please enter years of experience.")
    return predict_salary(years_experience)

with gr.Blocks(title="Salary Prediction") as demo:
    gr.Markdown("# Salary Prediction")
    gr.Markdown("Enter years of experience to estimate salary using Linear Regression.")
    years_input = gr.Number(label="Years of Experience", value=3.0, minimum=0)
    predict_button = gr.Button("Predict Salary")
    salary_output = gr.Number(label="Predicted Salary")
    predict_button.click(
        fn=gradio_predict,
        inputs=years_input,
        outputs=salary_output
    )

demo.launch(share=False)


## 7. Flask user interface
Run this cell after stopping the Gradio server if your notebook environment has trouble running both servers simultaneously. The Flask app opens at `http://127.0.0.1:5000` in local Jupyter.

In [ ]:
from flask import Flask, request, render_template_string

app = Flask("salary_prediction_app")

HTML = '''
<!doctype html>
<html>
<head>
  <meta charset="utf-8">
  <title>Salary Prediction</title>
  <style>
    body { font-family: Arial, sans-serif; max-width: 650px; margin: 50px auto; padding: 0 18px; }
    input, button { font-size: 16px; padding: 10px; margin-top: 8px; }
    input { width: 95%; }
    button { cursor: pointer; }
    .result { margin-top: 20px; padding: 14px; background: #e9f7ef; border-radius: 8px; }
    .error { margin-top: 20px; padding: 14px; background: #fdecea; border-radius: 8px; }
  </style>
</head>
<body>
  <h1>Salary Prediction</h1>
  <p>Linear Regression model trained on Salary_dataset.csv.</p>
  <form method="post">
    <label for="years">Years of Experience:</label><br>
    <input id="years" name="years" type="number" min="0" step="any"
           value="{{ years }}" required>
    <br><button type="submit">Predict Salary</button>
  </form>
  {% if prediction is not none %}
    <div class="result"><b>Estimated Salary:</b> {{ "{:,.2f}".format(prediction) }}</div>
  {% endif %}
  {% if error %}
    <div class="error"><b>Error:</b> {{ error }}</div>
  {% endif %}
</body>
</html>
'''

@app.route("/", methods=["GET", "POST"])
def home():
    years = "3"
    prediction = None
    error = None
    if request.method == "POST":
        years = request.form.get("years", "")
        try:
            prediction = predict_salary(years)
        except (TypeError, ValueError) as exc:
            error = str(exc)
    return render_template_string(
        HTML, years=years, prediction=prediction, error=error
    )

if __name__ == "__main__":
    app.run(host="127.0.0.1", port=5000, debug=False, use_reloader=False)


## 8. Conclusion
- `YearsExperience` is the independent variable (X).
- `Salary` is the dependent variable (y).
- Linear Regression learns a best-fit straight line and estimates salary for a given number of experience years.
- MAE and RMSE measure prediction error (lower is better); R² indicates how well the model explains variation in the test data.
- This small dataset is suitable for a learning demonstration; predictions are estimates, not guaranteed salaries.